In [8]:
import os
import argparse
import logging
import re
import warnings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.prompts import ChatPromptTemplate
from langchain_ollama import OllamaLLM
import streamlit as st
from rag import query_database, log_interaction


In [2]:
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [3]:
def get_embedding_function():
    return HuggingFaceEmbeddings(model_name="./all-MiniLM-L6-v2-local")


In [4]:
def populate_database():
    loader = TextLoader("processed_data.txt", encoding="utf-8")
    documents = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    texts = text_splitter.split_documents(documents)

    db = Chroma.from_documents(texts, get_embedding_function(), persist_directory="chroma_db")
    db.persist()
    print("Database built and persisted in chroma_db/")


In [5]:
logging.basicConfig(
    filename="chat_log.txt",
    level=logging.INFO,
    format="%(asctime)s - %(message)s"
)


In [ ]:
def log_interaction(question, answer):
    logging.info(f"User: {question}")
    logging.info(f"Bot: {answer}")
    logging.info("-----------------------------------------------------------------------------------------------------------------") 


In [7]:
def query_database(question: str):
    import logging as pylogging
    pylogging.getLogger("chromadb").setLevel(pylogging.ERROR)
    pylogging.getLogger("sentence_transformers").setLevel(pylogging.ERROR)
    pylogging.getLogger("httpx").setLevel(pylogging.ERROR)

    db = Chroma(persist_directory="chroma_db", embedding_function=get_embedding_function())
    retriever = db.as_retriever(search_kwargs={"k": 5})
    results = retriever.get_relevant_documents(question)

    context_chunks = [doc.page_content for doc in results]
    context = "\n".join(context_chunks)

    prompt_template = ChatPromptTemplate.from_template("""
    You are a scientific reasoning assistant.
    Use the retrieved knowledge to answer.
    If a gene is indirectly related to a process via GO relationships, explain the reasoning step by step.
    If no relation exists, say "No evidence found".

    Question: {question}
    Context:
    {context}
    Answer:
    """)

    llm = OllamaLLM(model="llama3")
    prompt = prompt_template.format(question=question, context=context)
    response = llm.invoke(prompt)

    return response.strip()



In [12]:
import logging
logging.getLogger("streamlit").setLevel(logging.ERROR)

In [ ]:
from IPython.display import display, Markdown
import ipywidgets as widgets


display(Markdown("# Gene Ontology RAG Chatbot"))


history = []


user_query = widgets.Text(
    value='',
    placeholder='Ask a question about a gene/GO term...',
    description='Query:',
    layout=widgets.Layout(width='80%')
)


ask_button = widgets.Button(description="Ask", button_style='primary')


output_area = widgets.Output()

def on_ask_clicked(b):
    query = user_query.value.strip()
    if query:
        with output_area:
            display(Markdown("Thinking..."))
        
        answer = query_database(query)  

        history.append((query, answer))
        log_interaction(query, answer)

        output_area.clear_output()
        with output_area:
            display(Markdown("## Chat History"))
            for i, (q, a) in enumerate(history, 1):
                display(Markdown(f"**Q{i}:** {q}"))
                display(Markdown(f"**A{i}:** {a}"))
                display(Markdown("---"))

ask_button.on_click(on_ask_clicked)


display(user_query, ask_button, output_area)


# Gene Ontology RAG Chatbot

Text(value='', description='Query:', layout=Layout(width='80%'), placeholder='Ask a question about a gene/GO t…

Button(button_style='primary', description='Ask', style=ButtonStyle())

Output()